# Breast Cancer Detection using VGG19 — Colab training

Fine-tunes a VGG19 classifier on the BreakHis 40X dataset.

**Before running:** upload the `Backend` folder from this project to `/content/Backend`
(drag it into the Colab file browser). Runtime → Change runtime type → **GPU**.

At the end, `vgg19_breakhis.pt` and `metrics.json` are downloaded; copy them into
`Backend/models/` on your laptop.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
%cd /content/Backend
!pip install -q scikit-learn matplotlib pillow reportlab

In [ ]:
# Download BreakHis (224 MB) and extract into data/breakhis
%cd /content/Backend
!mkdir -p data
!curl -L -o data/breakhis.zip https://huggingface.co/datasets/hirundo-io/BreaKHis-original/resolve/main/breakhis.zip
!unzip -q -o data/breakhis.zip -d data/ && ls data/breakhis

In [ ]:
# Patient-level 70/15/15 split
%cd /content/Backend
!python training/prepare_dataset.py --data-dir data/breakhis

In [ ]:
# Fine-tune VGG19 (last two blocks + head) on GPU
%cd /content/Backend
!python training/train.py --mode finetune --epochs 15 --batch-size 32 --lr 3e-4

In [ ]:
# Inspect the test metrics
import json
metrics = json.load(open('models/metrics.json'))
for key in ['mode', 'accuracy', 'accuracy_ci95', 'sensitivity', 'specificity', 'precision', 'f1_score', 'roc_auc', 'threshold', 'n_samples']:
    print(f'{key:14s}: {metrics.get(key)}')

In [ ]:
# Zip checkpoints + figures and download them
%cd /content/Backend
!zip -r /content/vgg19_breakhis_model.zip models/vgg19_breakhis.pt models/metrics.json models/history.json models/figures
from google.colab import files
files.download('/content/vgg19_breakhis_model.zip')